# ☁️ Ollama Cloud Runner — Google Colab GPU Backend

Run this notebook on **Google Colab** (with a **T4**, **L4**, or **A100** GPU runtime) to power your **Ollama Cloud Runner** web app over **ngrok**.

### Quick Setup
1. In Colab, click **Runtime → Change runtime type → T4 GPU** and click **Save**.
2. Get a free authtoken from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) and either:
   - Add it to **Colab Secrets** (🔑 icon on the left sidebar) named `NGROK_AUTHTOKEN`, **or**
   - Paste it when prompted in **Step 3** below.
3. Run all cells below, copy the printed **`https://....ngrok-free.app`** URL, and paste it into the **Ollama Cloud Runner** web app!

In [ ]:
# @title Step 1: Verify GPU & Install Backend Dependencies
!nvidia-smi || echo "⚠️ No GPU detected! Switch Runtime -> Change runtime type -> T4 GPU"
!apt-get update -qq && apt-get install -y -qq zstd pciutils lshw >/dev/null 2>&1 || true
!pip install -q fastapi "uvicorn[standard]" pyngrok httpx psutil beautifulsoup4 pydantic
print("✅ Dependencies installed!")

In [ ]:
# @title Step 2: Bootstrap Modular Backend Server Package
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/ollama_cloud_runner")
SERVER_PKG = PROJECT_ROOT / "server"
SERVER_PKG.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

SERVER_FILES = {'__init__.py': '"""Ollama Cloud Runner - Google Colab Backend Package."""\n', 'config.py': 'import os\nimport sys\nfrom pathlib import Path\n\nIS_LINUX = sys.platform.startswith("linux")\nIS_COLAB = (\n    "COLAB_GPU" in os.environ\n    or "COLAB_RELEASE_TAG" in os.environ\n    or Path("/content").exists()\n)\n\nDEFAULT_RUNNER_DIR = (\n    Path("/content/ollama_runner")\n    if IS_COLAB\n    else Path(os.getcwd()) / ".ollama_runner"\n)\n\nRUNNER_DIR = Path(os.environ.get("OLLAMA_RUNNER_DIR", str(DEFAULT_RUNNER_DIR)))\nRUNNER_DIR.mkdir(parents=True, exist_ok=True)\n\nRUNNER_SCRIPT_PATH = RUNNER_DIR / "start_ollama.sh"\nRUNNER_CONFIG_PATH = RUNNER_DIR / "runner_config.json"\nOLLAMA_LOG_PATH = RUNNER_DIR / "ollama_server.log"\n\nOLLAMA_HOST = os.environ.get("OLLAMA_INTERNAL_HOST", "http://127.0.0.1:11434")\nBACKEND_PORT = int(os.environ.get("PORT", "8000"))\n', 'models.py': 'from typing import List, Optional, Literal\nfrom pydantic import BaseModel, Field\n\n\nclass GpuMetrics(BaseModel):\n    available: bool = False\n    name: str = "No GPU Detected (CPU Runtime)"\n    vram_total_mb: int = 0\n    vram_used_mb: int = 0\n    utilization_percent: int = 0\n    driver_version: Optional[str] = None\n\n\nclass SystemMetrics(BaseModel):\n    cpu_percent: float = 0.0\n    ram_total_gb: float = 0.0\n    ram_used_gb: float = 0.0\n    disk_free_gb: float = 0.0\n    disk_total_gb: float = 0.0\n    platform_name: str = ""\n    is_colab: bool = False\n\n\nclass InstalledModel(BaseModel):\n    name: str\n    size_bytes: int = 0\n    size_gb: float = 0.0\n    modified_at: Optional[str] = None\n    family: Optional[str] = None\n    parameter_size: Optional[str] = None\n    quantization_level: Optional[str] = None\n\n\nclass ServerStatusResponse(BaseModel):\n    connected: bool = True\n    state: Literal["idle", "configuring", "starting", "pulling", "ready", "error"] = "idle"\n    state_message: str = "Colab agent connected and awaiting configuration"\n    ollama_installed: bool = False\n    ollama_running: bool = False\n    ollama_version: Optional[str] = None\n    configured_ollama_version: str = "latest"\n    active_model: Optional[str] = None\n    loaded_models: List[str] = Field(default_factory=list)\n    installed_models: List[InstalledModel] = Field(default_factory=list)\n    gpu: GpuMetrics\n    system: SystemMetrics\n    ngrok_url: Optional[str] = None\n    uptime_seconds: int = 0\n    last_updated_script_at: Optional[str] = None\n\n\nclass DeployRequest(BaseModel):\n    model: str = Field(..., description="Ollama model name and tag, e.g., llama3:8b")\n    ollama_version: str = Field(\n        default="latest",\n        description="Ollama version to install/run (e.g. \'latest\', \'0.6.2\', \'0.5.7\')",\n    )\n    num_ctx: int = Field(default=4096, ge=512, le=131072)\n    keep_alive: str = Field(default="15m")\n    flash_attention: bool = Field(default=True)\n    restart_server: bool = Field(default=True)\n\n\nclass ChatMessage(BaseModel):\n    role: Literal["system", "user", "assistant"]\n    content: str\n\n\nclass ChatRequest(BaseModel):\n    model: Optional[str] = None\n    messages: List[ChatMessage]\n    temperature: float = Field(default=0.7, ge=0.0, le=2.0)\n    num_ctx: int = Field(default=4096, ge=512, le=131072)\n    stream: bool = True\n', 'gpu_monitor.py': 'import platform\nimport shutil\nimport subprocess\nimport psutil\n\nfrom .config import IS_COLAB, RUNNER_DIR\nfrom .models import GpuMetrics, SystemMetrics\n\n\ndef get_gpu_metrics() -> GpuMetrics:\n    """Query NVIDIA GPU telemetry on Google Colab using nvidia-smi."""\n    try:\n        result = subprocess.run(\n            [\n                "nvidia-smi",\n                "--query-gpu=name,memory.total,memory.used,utilization.gpu,driver_version",\n                "--format=csv,noheader,nounits",\n            ],\n            capture_output=True,\n            text=True,\n            timeout=3,\n            check=False,\n        )\n        if result.returncode == 0 and result.stdout.strip():\n            first_line = result.stdout.strip().splitlines()[0]\n            parts = [p.strip() for p in first_line.split(",")]\n            if len(parts) >= 5:\n                return GpuMetrics(\n                    available=True,\n                    name=parts[0],\n                    vram_total_mb=int(float(parts[1])),\n                    vram_used_mb=int(float(parts[2])),\n                    utilization_percent=int(float(parts[3])),\n                    driver_version=parts[4],\n                )\n    except Exception:\n        pass\n\n    return GpuMetrics(available=False)\n\n\ndef get_system_metrics() -> SystemMetrics:\n    """Collect RAM, CPU, and Disk metrics for the Colab VM."""\n    vm = psutil.virtual_memory()\n    disk_target = str(RUNNER_DIR) if RUNNER_DIR.exists() else "/"\n    disk = shutil.disk_usage(disk_target)\n\n    return SystemMetrics(\n        cpu_percent=round(psutil.cpu_percent(interval=None), 1),\n        ram_total_gb=round(vm.total / (1024**3), 2),\n        ram_used_gb=round(vm.used / (1024**3), 2),\n        disk_free_gb=round(disk.free / (1024**3), 2),\n        disk_total_gb=round(disk.total / (1024**3), 2),\n        platform_name=f"{platform.system()} {platform.release()}",\n        is_colab=IS_COLAB,\n    )\n', 'catalog_scraper.py': 'import re\nimport time\nfrom typing import Any, Dict, List\nimport httpx\nfrom bs4 import BeautifulSoup\n\n_CATALOG_CACHE: Dict[str, Any] = {"timestamp": 0.0, "models": []}\nCACHE_TTL_SECONDS = 900  # 15 minutes\n\nCURATED_MODELS: List[Dict[str, Any]] = [\n    {\n        "name": "dolphin-llama3",\n        "title": "Dolphin 2.9 Llama 3 (Uncensored)",\n        "provider": "Eric Hartford / Cognitive Computations",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Flagship uncensored Llama 3 fine-tune with alignment and refusal guardrails removed. Available in 8B and full 70B for A100 GPUs.",\n        "tags": ["8b", "70b", "8b-256k"],\n        "default_tag": "70b",\n        "vram_hint": "42 GB (70b A100) / 5.5 GB (8b)",\n        "pulls": "4.8M+",\n        "capabilities": ["uncensored", "chat", "coding"],\n    },\n    {\n        "name": "huihui_ai/llama3.3-abliterated",\n        "title": "Llama 3.3 70B Abliterated (Uncensored A100)",\n        "provider": "Huihui AI (Meta Base)",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "State-of-the-art 70B Llama 3.3 with refusal vectors surgically removed via weight abliteration while keeping full 70B intelligence.",\n        "tags": ["70b", "latest"],\n        "default_tag": "70b",\n        "vram_hint": "43 GB VRAM (A100)",\n        "pulls": "Community Top",\n        "capabilities": ["uncensored", "chat", "tools"],\n    },\n    {\n        "name": "huihui_ai/deepseek-r1-abliterated",\n        "title": "DeepSeek-R1 Abliterated (Uncensored Reasoning)",\n        "provider": "Huihui AI (DeepSeek Base)",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Uncensored DeepSeek-R1 chain-of-thought reasoning model with refusal triggers abliterated across 8B, 14B, 32B, and 70B A100 sizes.",\n        "tags": ["8b", "14b", "32b", "70b"],\n        "default_tag": "32b",\n        "vram_hint": "20 GB (32b) / 43 GB (70b)",\n        "pulls": "Community Top",\n        "capabilities": ["uncensored", "reasoning"],\n    },\n    {\n        "name": "dolphin-mixtral",\n        "title": "Dolphin 2.7 Mixtral MoE (8x7B & 8x22B Uncensored)",\n        "provider": "Eric Hartford",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Uncensored Mixture-of-Experts powerhouse trained on coding, roleplay, and unfiltered instruction datasets. Built for high-VRAM A100s.",\n        "tags": ["8x7b", "8x22b"],\n        "default_tag": "8x7b",\n        "vram_hint": "27 GB (8x7b) / 80 GB (8x22b)",\n        "pulls": "3.6M+",\n        "capabilities": ["uncensored", "MoE", "coding"],\n    },\n    {\n        "name": "hermes3",\n        "title": "Nous Hermes 3 (8B / 70B Unfiltered)",\n        "provider": "Nous Research",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Nous Research flagship unfiltered Llama 3.1 model engineered for neutral compliance, deep roleplay, function calling, and reasoning.",\n        "tags": ["3b", "8b", "70b"],\n        "default_tag": "70b",\n        "vram_hint": "42 GB (70b A100) / 5.5 GB (8b)",\n        "pulls": "2.4M+",\n        "capabilities": ["uncensored", "chat", "tools"],\n    },\n    {\n        "name": "huihui_ai/qwen2.5-abliterated",\n        "title": "Qwen 2.5 Abliterated (Up to 72B Uncensored)",\n        "provider": "Huihui AI (Alibaba Base)",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Uncensored Qwen 2.5 series with refusal directions removed from activation space. Available up to massive 32B and 72B checkpoints.",\n        "tags": ["7b", "14b", "32b", "72b"],\n        "default_tag": "32b",\n        "vram_hint": "20 GB (32b) / 45 GB (72b)",\n        "pulls": "Community Top",\n        "capabilities": ["uncensored", "chat"],\n    },\n    {\n        "name": "huihui_ai/qwen2.5-coder-abliterated",\n        "title": "Qwen 2.5 Coder Abliterated (Uncensored Code)",\n        "provider": "Huihui AI",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Uncensored coding, reverse-engineering, and security auditing model without code-generation refusals. Up to 32B parameters.",\n        "tags": ["7b", "14b", "32b"],\n        "default_tag": "32b",\n        "vram_hint": "20 GB (32b A100) / 9.5 GB (14b)",\n        "pulls": "Community Top",\n        "capabilities": ["uncensored", "coding"],\n    },\n    {\n        "name": "huihui_ai/gemma3-abliterated",\n        "title": "Gemma 3 Abliterated (27B Uncensored)",\n        "provider": "Huihui AI (Google Base)",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Google Gemma 3 with safety refusal vectors abliterated. The 27B checkpoint runs effortlessly on a single Colab A100.",\n        "tags": ["4b", "12b", "27b"],\n        "default_tag": "27b",\n        "vram_hint": "18 GB VRAM (27b)",\n        "pulls": "Community Top",\n        "capabilities": ["uncensored", "vision", "chat"],\n    },\n    {\n        "name": "dolphin3",\n        "title": "Dolphin 3.0 (Llama 3.1 8B Uncensored)",\n        "provider": "Cognitive Computations",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "description": "Latest generation Dolphin 3.0 steering-controlled uncensored model giving full system-prompt control to the user.",\n        "tags": ["8b", "latest"],\n        "default_tag": "8b",\n        "vram_hint": "5.5 GB VRAM (8b)",\n        "pulls": "1.1M+",\n        "capabilities": ["uncensored", "chat", "coding"],\n    },\n    {\n        "name": "wizard-vicuna-uncensored",\n        "title": "Wizard Vicuna Uncensored (7B / 13B / 30B)",\n        "provider": "Eric Hartford",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Classic Evol-Instruct uncensored model trained against a dataset stripped of alignment lectures and refusals.",\n        "tags": ["7b", "13b", "30b"],\n        "default_tag": "30b",\n        "vram_hint": "19 GB (30b A100) / 8.5 GB (13b)",\n        "pulls": "2.9M+",\n        "capabilities": ["uncensored", "chat"],\n    },\n    {\n        "name": "nous-hermes2-mixtral",\n        "title": "Nous Hermes 2 Mixtral 8x7B (Unfiltered MoE)",\n        "provider": "Nous Research",\n        "category": "uncensored",\n        "featured": True,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "High-VRAM 8x7B Mixture-of-Experts model by Nous Research trained on 1M+ primarily synthetic unfiltered instructions.",\n        "tags": ["8x7b", "dpo"],\n        "default_tag": "8x7b",\n        "vram_hint": "27 GB VRAM (A100)",\n        "pulls": "1.7M+",\n        "capabilities": ["uncensored", "MoE"],\n    },\n    {\n        "name": "dolphin-mistral",\n        "title": "Dolphin 2.8 Mistral 7B (Uncensored)",\n        "provider": "Eric Hartford",\n        "category": "uncensored",\n        "featured": False,\n        "uncensored": True,\n        "description": "Fast, uninhibited Mistral 7B fine-tune that obeys any system prompt without refusals.",\n        "tags": ["7b", "latest"],\n        "default_tag": "7b",\n        "vram_hint": "4.8 GB VRAM (7b)",\n        "pulls": "3.4M+",\n        "capabilities": ["uncensored", "chat"],\n    },\n    {\n        "name": "llama2-uncensored",\n        "title": "Llama 2 Uncensored (7B & 70B)",\n        "provider": "George Sung / Eric Hartford",\n        "category": "uncensored",\n        "featured": False,\n        "uncensored": True,\n        "high_vram": True,\n        "description": "Uncensored Llama 2 7B and 70B models fine-tuned on unfiltered conversation datasets.",\n        "tags": ["7b", "70b"],\n        "default_tag": "70b",\n        "vram_hint": "40 GB (70b A100) / 4.5 GB (7b)",\n        "pulls": "4.2M+",\n        "capabilities": ["uncensored", "chat"],\n    },\n    {\n        "name": "wizardlm-uncensored",\n        "title": "WizardLM Uncensored 13B",\n        "provider": "Eric Hartford",\n        "category": "uncensored",\n        "featured": False,\n        "uncensored": True,\n        "description": "Uncensored 13B WizardLM variant for unrestricted complex instruction following.",\n        "tags": ["13b", "latest"],\n        "default_tag": "13b",\n        "vram_hint": "8.5 GB VRAM (13b)",\n        "pulls": "1.5M+",\n        "capabilities": ["uncensored", "chat"],\n    },\n    {\n        "name": "samantha-mistral",\n        "title": "Samantha Mistral (Uncensored Companion)",\n        "provider": "Eric Hartford",\n        "category": "uncensored",\n        "featured": False,\n        "uncensored": True,\n        "description": "Uncensored philosophy, psychology, and personal companion model trained on Mistral 7B.",\n        "tags": ["7b", "latest"],\n        "default_tag": "7b",\n        "vram_hint": "4.8 GB VRAM (7b)",\n        "pulls": "950K+",\n        "capabilities": ["uncensored", "chat"],\n    },\n    {\n        "name": "llama3.3",\n        "title": "Llama 3.3 70B",\n        "provider": "Meta",\n        "category": "general",\n        "featured": True,\n        "high_vram": True,\n        "description": "High-efficiency 70B instruction-tuned model approaching 405B performance on A100 Colab.",\n        "tags": ["70b"],\n        "default_tag": "70b",\n        "vram_hint": "43 GB (70b)",\n        "pulls": "3.1M+",\n        "capabilities": ["chat", "tools"],\n    },\n    {\n        "name": "llama3",\n        "title": "Llama 3",\n        "provider": "Meta",\n        "category": "general",\n        "featured": True,\n        "high_vram": True,\n        "description": "Meta Llama 3 — flagship open weights LLM optimized for dialogue, reasoning, and general tasks.",\n        "tags": ["8b", "70b", "instruct"],\n        "default_tag": "8b",\n        "vram_hint": "5.5 GB (8b) / 42 GB (70b)",\n        "pulls": "12M+",\n        "capabilities": ["chat", "instruct"],\n    },\n    {\n        "name": "deepseek-r1",\n        "title": "DeepSeek-R1",\n        "provider": "DeepSeek",\n        "category": "reasoning",\n        "featured": True,\n        "high_vram": True,\n        "description": "Open reasoning models across 1.5B, 7B, 8B, 14B, 32B, and 70B sizes.",\n        "tags": ["1.5b", "7b", "8b", "14b", "32b", "70b"],\n        "default_tag": "32b",\n        "vram_hint": "20 GB (32b) / 43 GB (70b)",\n        "pulls": "22M+",\n        "capabilities": ["reasoning", "chat"],\n    },\n    {\n        "name": "qwen2.5",\n        "title": "Qwen 2.5",\n        "provider": "Alibaba",\n        "category": "general",\n        "featured": True,\n        "high_vram": True,\n        "description": "Dense multilingual instruction models from 3B up to 72B with structured JSON and tool support.",\n        "tags": ["3b", "7b", "14b", "32b", "72b"],\n        "default_tag": "32b",\n        "vram_hint": "20 GB (32b) / 45 GB (72b)",\n        "pulls": "11M+",\n        "capabilities": ["chat", "tools"],\n    },\n]\n\n\nasync def fetch_ollama_library_catalog() -> List[Dict[str, Any]]:\n    """Fetch all available models from ollama.com/library and merge with curated uncensored metadata."""\n    now = time.time()\n    if _CATALOG_CACHE["models"] and (now - _CATALOG_CACHE["timestamp"]) < CACHE_TTL_SECONDS:\n        return _CATALOG_CACHE["models"]\n\n    curated_by_name = {m["name"]: dict(m) for m in CURATED_MODELS}\n    discovered: List[Dict[str, Any]] = []\n\n    try:\n        async with httpx.AsyncClient(timeout=8.0, follow_redirects=True) as client:\n            resp = await client.get(\n                "https://ollama.com/library",\n                headers={"User-Agent": "Mozilla/5.0 OllamaCloudRunner/1.0"},\n            )\n            if resp.status_code == 200:\n                soup = BeautifulSoup(resp.text, "html.parser")\n                for link in soup.select(\'a[href^="/library/"]\'):\n                    href = link.get("href", "")\n                    slug = href.replace("/library/", "").strip("/").split("/")[0]\n                    if not slug or ":" in slug:\n                        continue\n\n                    if slug in curated_by_name:\n                        discovered.append(curated_by_name.pop(slug))\n                        continue\n\n                    desc_el = link.find("p")\n                    description = (\n                        desc_el.get_text(" ", strip=True)\n                        if desc_el\n                        else f"Official {slug} model from the Ollama library."\n                    )\n\n                    badge_texts = [\n                        s.get_text(strip=True)\n                        for s in link.find_all("span")\n                        if s.get_text(strip=True)\n                    ]\n                    size_tags = [\n                        b.lower()\n                        for b in badge_texts\n                        if re.match(r"^\\d+(x\\d+)?(\\.\\d+)?[bm]$|^e\\d+b$", b.lower())\n                    ]\n                    capabilities = [\n                        b.lower()\n                        for b in badge_texts\n                        if b.lower() in {"vision", "tools", "thinking", "embedding", "code"}\n                    ]\n                    if not size_tags:\n                        size_tags = ["latest"]\n\n                    is_uncensored = any(\n                        kw in slug.lower() or kw in description.lower()\n                        for kw in ("uncensored", "dolphin", "abliterated", "hermes", "samantha")\n                    )\n                    is_high_vram = any(\n                        t in size_tags for t in ("27b", "30b", "32b", "34b", "70b", "72b", "8x7b", "8x22b")\n                    )\n\n                    category = "uncensored" if is_uncensored else "general"\n                    if not is_uncensored:\n                        if "embedding" in capabilities or "embed" in slug:\n                            category = "embedding"\n                        elif "vision" in capabilities or "vl" in slug:\n                            category = "vision"\n                        elif "thinking" in capabilities or "r1" in slug:\n                            category = "reasoning"\n                        elif "coder" in slug or "code" in slug:\n                            category = "coding"\n\n                    discovered.append(\n                        {\n                            "name": slug,\n                            "title": slug.replace("-", " ").title(),\n                            "provider": "Ollama Library",\n                            "category": category,\n                            "featured": False,\n                            "uncensored": is_uncensored,\n                            "high_vram": is_high_vram,\n                            "description": description,\n                            "tags": size_tags[:8],\n                            "default_tag": size_tags[-1] if is_high_vram else size_tags[0],\n                            "vram_hint": "Auto",\n                            "pulls": "Library",\n                            "capabilities": (["uncensored"] if is_uncensored else []) + (capabilities or ["chat"]),\n                        }\n                    )\n    except Exception:\n        pass\n\n    merged = list(CURATED_MODELS)\n    seen = {m["name"] for m in merged}\n    for item in discovered:\n        if item["name"] not in seen:\n            merged.append(item)\n            seen.add(item["name"])\n\n    _CATALOG_CACHE["timestamp"] = now\n    _CATALOG_CACHE["models"] = merged\n    return merged\n', 'ollama_manager.py': 'import asyncio\nimport json\nimport os\nimport shutil\nimport subprocess\nimport time\nfrom datetime import datetime, timezone\nfrom typing import Any, AsyncGenerator, Dict, List, Optional\nimport httpx\n\nfrom .config import (\n    IS_LINUX,\n    OLLAMA_HOST,\n    OLLAMA_LOG_PATH,\n    RUNNER_CONFIG_PATH,\n    RUNNER_DIR,\n    RUNNER_SCRIPT_PATH,\n)\nfrom .models import ChatRequest, DeployRequest, InstalledModel\n\n\nclass OllamaManager:\n    """Manages Ollama installation, generated server code, process lifecycle, and model pulls."""\n\n    def __init__(self) -> None:\n        self.process: Optional[subprocess.Popen] = None\n        self.state: str = "idle"\n        self.state_message: str = "Colab agent ready — select a model to update & start server"\n        self.active_model: Optional[str] = None\n        self.configured_ollama_version: str = "latest"\n        self.last_updated_script_at: Optional[str] = None\n        self.started_at: float = time.time()\n        self._load_existing_config()\n\n    def _load_existing_config(self) -> None:\n        if RUNNER_CONFIG_PATH.exists():\n            try:\n                data = json.loads(RUNNER_CONFIG_PATH.read_text(encoding="utf-8"))\n                self.active_model = data.get("model")\n                self.configured_ollama_version = data.get("ollama_version", "latest")\n                self.last_updated_script_at = data.get("updated_at")\n            except Exception:\n                pass\n        else:\n            self._write_runner_files(\n                DeployRequest(\n                    model="llama3:8b",\n                    ollama_version="latest",\n                    num_ctx=4096,\n                    keep_alive="15m",\n                    flash_attention=True,\n                ),\n                initial=True,\n            )\n\n    def _write_runner_files(self, req: DeployRequest, initial: bool = False) -> Dict[str, Any]:\n        """Generate and overwrite the Colab runner shell script and JSON configuration."""\n        now_iso = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")\n        config_payload = {\n            "model": req.model,\n            "ollama_version": req.ollama_version,\n            "num_ctx": req.num_ctx,\n            "keep_alive": req.keep_alive,\n            "flash_attention": req.flash_attention,\n            "ollama_host": "127.0.0.1:11434",\n            "updated_at": now_iso,\n        }\n        RUNNER_CONFIG_PATH.write_text(\n            json.dumps(config_payload, indent=2), encoding="utf-8"\n        )\n\n        version_env = (\n            f\'OLLAMA_VERSION="{req.ollama_version}" \'\n            if req.ollama_version and req.ollama_version != "latest"\n            else ""\n        )\n        flash_val = "1" if req.flash_attention else "0"\n\n        script_content = f"""#!/usr/bin/env bash\n# ==============================================================================\n# Auto-Generated by Ollama Cloud Runner (Google Colab Backend)\n# Updated At : {now_iso}\n# Target Model: {req.model}\n# Ollama Ver  : {req.ollama_version}\n# ==============================================================================\nset -euo pipefail\n\nexport OLLAMA_HOST="127.0.0.1:11434"\nexport OLLAMA_ORIGINS="*"\nexport OLLAMA_KEEP_ALIVE="{req.keep_alive}"\nexport OLLAMA_FLASH_ATTENTION="{flash_val}"\nexport OLLAMA_NUM_CTX="{req.num_ctx}"\n\nif ! command -v ollama >/dev/null 2>&1; then\n  echo "[Runner] Installing Ollama ({req.ollama_version})..."\n  curl -fsSL https://ollama.com/install.sh | {version_env}sh\nfi\n\necho "[Runner] Starting Ollama server on $OLLAMA_HOST..."\nnohup ollama serve > "{OLLAMA_LOG_PATH.as_posix()}" 2>&1 &\n\nfor i in $(seq 1 30); do\n  if curl -sf http://127.0.0.1:11434/api/version >/dev/null 2>&1; then\n    echo "[Runner] Ollama daemon is live!"\n    break\n  fi\n  sleep 1\ndone\n\necho "[Runner] Pulling model {req.model}..."\nollama pull "{req.model}"\necho "[Runner] Model {req.model} ready for ngrok frontend requests."\n"""\n        RUNNER_SCRIPT_PATH.write_text(script_content, encoding="utf-8")\n        if IS_LINUX:\n            try:\n                os.chmod(RUNNER_SCRIPT_PATH, 0o755)\n            except Exception:\n                pass\n\n        if not initial:\n            self.last_updated_script_at = now_iso\n            self.configured_ollama_version = req.ollama_version\n\n        return config_payload\n\n    def get_runner_code_and_logs(self) -> Dict[str, Any]:\n        script = (\n            RUNNER_SCRIPT_PATH.read_text(encoding="utf-8")\n            if RUNNER_SCRIPT_PATH.exists()\n            else ""\n        )\n        config_raw = (\n            RUNNER_CONFIG_PATH.read_text(encoding="utf-8")\n            if RUNNER_CONFIG_PATH.exists()\n            else "{}"\n        )\n        logs = ""\n        if OLLAMA_LOG_PATH.exists():\n            try:\n                lines = OLLAMA_LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()\n                logs = "\\n".join(lines[-80:])\n            except Exception:\n                logs = ""\n        return {\n            "script_path": str(RUNNER_SCRIPT_PATH),\n            "config_path": str(RUNNER_CONFIG_PATH),\n            "log_path": str(OLLAMA_LOG_PATH),\n            "script": script,\n            "config": json.loads(config_raw) if config_raw else {},\n            "logs": logs,\n            "last_updated_at": self.last_updated_script_at,\n        }\n\n    def is_ollama_installed(self) -> bool:\n        return shutil.which("ollama") is not None\n\n    async def get_ollama_version(self) -> Optional[str]:\n        try:\n            async with httpx.AsyncClient(timeout=2.0) as client:\n                resp = await client.get(f"{OLLAMA_HOST}/api/version")\n                if resp.status_code == 200:\n                    return resp.json().get("version")\n        except Exception:\n            pass\n\n        if self.is_ollama_installed():\n            try:\n                proc = await asyncio.create_subprocess_exec(\n                    "ollama",\n                    "--version",\n                    stdout=asyncio.subprocess.PIPE,\n                    stderr=asyncio.subprocess.PIPE,\n                )\n                out, _ = await asyncio.wait_for(proc.communicate(), timeout=3.0)\n                text = out.decode("utf-8", errors="ignore").strip()\n                if text:\n                    return text.split()[-1]\n            except Exception:\n                pass\n        return None\n\n    async def is_ollama_running(self) -> bool:\n        try:\n            async with httpx.AsyncClient(timeout=1.5) as client:\n                resp = await client.get(f"{OLLAMA_HOST}/api/version")\n                return resp.status_code == 200\n        except Exception:\n            return False\n\n    async def get_installed_models(self) -> List[InstalledModel]:\n        try:\n            async with httpx.AsyncClient(timeout=3.0) as client:\n                resp = await client.get(f"{OLLAMA_HOST}/api/tags")\n                if resp.status_code == 200:\n                    raw_models = resp.json().get("models", [])\n                    results: List[InstalledModel] = []\n                    for m in raw_models:\n                        size_b = int(m.get("size", 0))\n                        details = m.get("details", {}) or {}\n                        results.append(\n                            InstalledModel(\n                                name=m.get("name", ""),\n                                size_bytes=size_b,\n                                size_gb=round(size_b / (1024**3), 2),\n                                modified_at=m.get("modified_at"),\n                                family=details.get("family"),\n                                parameter_size=details.get("parameter_size"),\n                                quantization_level=details.get("quantization_level"),\n                            )\n                        )\n                    return results\n        except Exception:\n            pass\n        return []\n\n    async def get_loaded_models(self) -> List[str]:\n        try:\n            async with httpx.AsyncClient(timeout=2.0) as client:\n                resp = await client.get(f"{OLLAMA_HOST}/api/ps")\n                if resp.status_code == 200:\n                    return [m.get("name", "") for m in resp.json().get("models", []) if m.get("name")]\n        except Exception:\n            pass\n        return []\n\n    async def stop_ollama(self) -> Dict[str, Any]:\n        if self.process and self.process.poll() is None:\n            self.process.terminate()\n            try:\n                self.process.wait(timeout=4)\n            except subprocess.TimeoutExpired:\n                self.process.kill()\n        if IS_LINUX:\n            subprocess.run(["pkill", "-f", "ollama serve"], check=False)\n        self.state = "idle"\n        self.state_message = "Ollama server stopped"\n        return {"stopped": True}\n\n    async def delete_model(self, model_name: str) -> Dict[str, Any]:\n        async with httpx.AsyncClient(timeout=10.0) as client:\n            resp = await client.request(\n                "DELETE", f"{OLLAMA_HOST}/api/delete", json={"name": model_name}\n            )\n            if resp.status_code == 200:\n                if self.active_model == model_name:\n                    self.active_model = None\n                return {"deleted": True, "model": model_name}\n            return {"deleted": False, "detail": resp.text}\n\n    async def deploy_and_start(self, req: DeployRequest) -> AsyncGenerator[str, None]:\n        """\n        Stream NDJSON progress events as the Colab server updates its runner code,\n        installs/updates Ollama if needed, starts `ollama serve`, pulls the model,\n        and warms it up on the GPU.\n        """\n        def emit(step: str, message: str, progress: Optional[int] = None, **extra: Any) -> str:\n            payload = {"step": step, "message": message, "progress": progress, **extra}\n            return json.dumps(payload) + "\\n"\n\n        try:\n            # Step 1: Update Colab runner script & configuration\n            self.state = "configuring"\n            self.state_message = f"Updating Colab server runner code for {req.model}..."\n            config_written = self._write_runner_files(req, initial=False)\n            yield emit(\n                "code_updated",\n                f"Updated Colab server code ({RUNNER_SCRIPT_PATH.name}) for model \'{req.model}\' (Ollama {req.ollama_version})",\n                progress=15,\n                config=config_written,\n            )\n            await asyncio.sleep(0.25)\n\n            # Step 2: Install or verify Ollama binary version on Linux/Colab\n            current_ver = await self.get_ollama_version()\n            need_install = not self.is_ollama_installed()\n            need_version_change = (\n                IS_LINUX\n                and req.ollama_version != "latest"\n                and current_ver is not None\n                and req.ollama_version not in current_ver\n            )\n\n            if need_install or need_version_change:\n                if not IS_LINUX:\n                    yield emit(\n                        "error",\n                        "Ollama is not installed on this non-Linux machine. Please install Ollama locally or run this backend on Google Colab.",\n                        progress=0,\n                    )\n                    self.state = "error"\n                    return\n\n                self.state = "starting"\n                ver_label = req.ollama_version\n                self.state_message = f"Installing Ollama ({ver_label}) on Colab VM..."\n                yield emit(\n                    "installing_ollama",\n                    f"Installing Ollama ({ver_label}) on Google Colab...",\n                    progress=25,\n                )\n                # Ensure zstd / pciutils exist on Ubuntu Colab images\n                subprocess.run(\n                    "apt-get update -qq && apt-get install -y -qq zstd pciutils lshw >/dev/null 2>&1 || true",\n                    shell=True,\n                    check=False,\n                )\n                env = os.environ.copy()\n                if req.ollama_version and req.ollama_version != "latest":\n                    env["OLLAMA_VERSION"] = req.ollama_version\n\n                install_proc = await asyncio.create_subprocess_shell(\n                    "curl -fsSL https://ollama.com/install.sh | sh",\n                    env=env,\n                    stdout=asyncio.subprocess.PIPE,\n                    stderr=asyncio.subprocess.STDOUT,\n                )\n                assert install_proc.stdout is not None\n                async for raw_line in install_proc.stdout:\n                    line = raw_line.decode("utf-8", errors="ignore").strip()\n                    if line:\n                        yield emit("install_log", line, progress=30)\n                await install_proc.wait()\n            else:\n                yield emit(\n                    "ollama_verified",\n                    f"Ollama engine verified (v{current_ver or \'installed\'})",\n                    progress=30,\n                )\n\n            # Step 3: Start (or restart) `ollama serve` with updated environment variables\n            self.state = "starting"\n            is_running = await self.is_ollama_running()\n            if is_running and req.restart_server:\n                yield emit("restarting_daemon", "Applying updated server configuration to Ollama daemon...", progress=38)\n                await self.stop_ollama()\n                await asyncio.sleep(0.8)\n                is_running = False\n\n            if not is_running:\n                self.state_message = "Starting Ollama GPU server daemon..."\n                yield emit("starting_daemon", "Starting `ollama serve` on Colab GPU...", progress=42)\n                serve_env = os.environ.copy()\n                serve_env["OLLAMA_HOST"] = "127.0.0.1:11434"\n                serve_env["OLLAMA_ORIGINS"] = "*"\n                serve_env["OLLAMA_KEEP_ALIVE"] = req.keep_alive\n                serve_env["OLLAMA_FLASH_ATTENTION"] = "1" if req.flash_attention else "0"\n\n                log_file = open(OLLAMA_LOG_PATH, "a", encoding="utf-8")\n                self.process = subprocess.Popen(\n                    ["ollama", "serve"],\n                    env=serve_env,\n                    stdout=log_file,\n                    stderr=subprocess.STDOUT,\n                    start_new_session=IS_LINUX,\n                )\n\n                daemon_up = False\n                for _ in range(30):\n                    if await self.is_ollama_running():\n                        daemon_up = True\n                        break\n                    await asyncio.sleep(0.5)\n\n                if not daemon_up:\n                    self.state = "error"\n                    self.state_message = "Failed to start `ollama serve` within 15s"\n                    yield emit("error", "Timed out waiting for `ollama serve` on port 11434.", progress=42)\n                    return\n\n            live_ver = await self.get_ollama_version()\n            yield emit(\n                "daemon_ready",\n                f"Ollama server running on Colab (v{live_ver or \'active\'})",\n                progress=50,\n            )\n\n            # Step 4: Pull requested model with streaming progress\n            self.state = "pulling"\n            self.state_message = f"Pulling model {req.model} onto Colab..."\n            yield emit("pull_start", f"Pulling \'{req.model}\' from Ollama registry...", progress=52)\n\n            last_emit_ts = 0.0\n            async with httpx.AsyncClient(timeout=None) as client:\n                async with client.stream(\n                    "POST",\n                    f"{OLLAMA_HOST}/api/pull",\n                    json={"name": req.model, "stream": True},\n                ) as pull_resp:\n                    if pull_resp.status_code != 200:\n                        err_body = (await pull_resp.aread()).decode("utf-8", errors="ignore")\n                        self.state = "error"\n                        self.state_message = f"Failed to pull {req.model}"\n                        yield emit("error", f"Ollama pull failed: {err_body}", progress=52)\n                        return\n\n                    async for line in pull_resp.aiter_lines():\n                        if not line.strip():\n                            continue\n                        try:\n                            evt = json.loads(line)\n                        except json.JSONDecodeError:\n                            continue\n\n                        if "error" in evt:\n                            self.state = "error"\n                            self.state_message = evt["error"]\n                            yield emit("error", f"Model error: {evt[\'error\']}", progress=52)\n                            return\n\n                        status_txt = evt.get("status", "Downloading...")\n                        total = evt.get("total", 0)\n                        completed = evt.get("completed", 0)\n                        now = time.time()\n\n                        if total and total > 0:\n                            ratio = min(1.0, completed / total)\n                            overall_pct = int(52 + ratio * 40)\n                            dl_pct = round(ratio * 100, 1)\n                            if (now - last_emit_ts) >= 0.25 or completed >= total:\n                                last_emit_ts = now\n                                mb_done = round(completed / (1024**2), 1)\n                                mb_tot = round(total / (1024**2), 1)\n                                yield emit(\n                                    "pull_progress",\n                                    f"{status_txt}: {mb_done} MB / {mb_tot} MB ({dl_pct}%)",\n                                    progress=overall_pct,\n                                    download_percent=dl_pct,\n                                )\n                        else:\n                            if (now - last_emit_ts) >= 0.2 or status_txt == "success":\n                                last_emit_ts = now\n                                yield emit("pull_status", status_txt, progress=92)\n\n            # Step 5: Warm up model in GPU VRAM with matching num_ctx so chat doesn\'t trigger a reload\n            yield emit("warming_up", f"Loading \'{req.model}\' into Colab GPU memory (ctx={req.num_ctx})...", progress=95)\n            try:\n                async with httpx.AsyncClient(timeout=180.0) as client:\n                    await client.post(\n                        f"{OLLAMA_HOST}/api/chat",\n                        json={\n                            "model": req.model,\n                            "messages": [],\n                            "keep_alive": req.keep_alive,\n                            "options": {\n                                "num_ctx": req.num_ctx,\n                            },\n                        },\n                    )\n            except Exception:\n                pass\n\n            self.active_model = req.model\n            self.state = "ready"\n            self.state_message = f"Active & serving {req.model}"\n            yield emit(\n                "ready",\n                f"Colab server updated and serving \'{req.model}\' via ngrok!",\n                progress=100,\n                active_model=req.model,\n            )\n\n        except Exception as exc:\n            self.state = "error"\n            self.state_message = str(exc)\n            yield emit("error", f"Deployment error: {exc}", progress=0)\n\n    async def stream_chat(self, req: ChatRequest) -> AsyncGenerator[str, None]:\n        target_model = req.model or self.active_model\n        if not target_model:\n            yield json.dumps({"error": "No active model selected on Colab server."}) + "\\n"\n            return\n\n        # Flush initial event immediately through FastAPI & ngrok\n        yield json.dumps({"status": f"Running inference on {target_model}..."}) + "\\n"\n\n        ollama_payload = {\n            "model": target_model,\n            "messages": [m.model_dump() for m in req.messages],\n            "stream": True,\n            "keep_alive": "15m",\n            "options": {\n                "temperature": req.temperature,\n                "num_ctx": req.num_ctx,\n            },\n        }\n\n        try:\n            async with httpx.AsyncClient(timeout=None) as client:\n                async with client.stream(\n                    "POST", f"{OLLAMA_HOST}/api/chat", json=ollama_payload\n                ) as resp:\n                    if resp.status_code != 200:\n                        body = (await resp.aread()).decode("utf-8", errors="ignore")\n                        yield json.dumps({"error": f"Ollama error ({resp.status_code}): {body}"}) + "\\n"\n                        return\n                    async for line in resp.aiter_lines():\n                        if line.strip():\n                            yield line + "\\n"\n        except Exception as exc:\n            yield json.dumps({"error": str(exc)}) + "\\n"\n', 'main.py': 'import argparse\nimport os\nimport time\nfrom typing import Optional\nimport uvicorn\nfrom fastapi import FastAPI, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import StreamingResponse\n\nfrom .catalog_scraper import fetch_ollama_library_catalog\nfrom .config import BACKEND_PORT\nfrom .gpu_monitor import get_gpu_metrics, get_system_metrics\nfrom .models import ChatRequest, DeployRequest, ServerStatusResponse\nfrom .ollama_manager import OllamaManager\n\napp = FastAPI(\n    title="Ollama Cloud Runner - Google Colab Backend",\n    description="Control agent and API gateway bridging Google Colab GPUs to the Ollama Cloud Runner frontend via ngrok.",\n    version="1.0.0",\n)\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=False,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\nmanager = OllamaManager()\nPUBLIC_NGROK_URL: Optional[str] = os.environ.get("PUBLIC_NGROK_URL")\n\n\n@app.get("/")\nasync def root():\n    return {\n        "service": "Ollama Cloud Runner (Google Colab Backend)",\n        "connected": True,\n        "ngrok_url": PUBLIC_NGROK_URL,\n        "active_model": manager.active_model,\n    }\n\n\n@app.get("/api/status", response_model=ServerStatusResponse)\nasync def get_server_status():\n    ollama_running = await manager.is_ollama_running()\n    ollama_version = await manager.get_ollama_version()\n    installed = await manager.get_installed_models() if ollama_running else []\n    loaded = await manager.get_loaded_models() if ollama_running else []\n\n    active = manager.active_model\n    if not active and loaded:\n        active = loaded[0]\n\n    state = manager.state\n    if ollama_running and state == "idle" and active:\n        state = "ready"\n\n    return ServerStatusResponse(\n        connected=True,\n        state=state,  # type: ignore[arg-type]\n        state_message=manager.state_message,\n        ollama_installed=manager.is_ollama_installed(),\n        ollama_running=ollama_running,\n        ollama_version=ollama_version,\n        configured_ollama_version=manager.configured_ollama_version,\n        active_model=active,\n        loaded_models=loaded,\n        installed_models=installed,\n        gpu=get_gpu_metrics(),\n        system=get_system_metrics(),\n        ngrok_url=PUBLIC_NGROK_URL,\n        uptime_seconds=int(time.time() - manager.started_at),\n        last_updated_script_at=manager.last_updated_script_at,\n    )\n\n\n@app.get("/api/models/catalog")\nasync def list_catalog():\n    catalog = await fetch_ollama_library_catalog()\n    installed = await manager.get_installed_models()\n    installed_names = {m.name for m in installed}\n    installed_bases = {m.name.split(":")[0] for m in installed}\n\n    enriched = []\n    for item in catalog:\n        copy_item = dict(item)\n        copy_item["is_installed"] = (\n            item["name"] in installed_bases or f"{item[\'name\']}:{item[\'default_tag\']}" in installed_names\n        )\n        enriched.append(copy_item)\n\n    return {\n        "count": len(enriched),\n        "installed": [m.model_dump() for m in installed],\n        "models": enriched,\n    }\n\n\n@app.get("/api/models/installed")\nasync def list_installed_models():\n    installed = await manager.get_installed_models()\n    loaded = await manager.get_loaded_models()\n    return {\n        "installed": [m.model_dump() for m in installed],\n        "loaded": loaded,\n    }\n\n\n@app.delete("/api/models/{model_name:path}")\nasync def remove_model(model_name: str):\n    if not await manager.is_ollama_running():\n        raise HTTPException(status_code=400, detail="Ollama server is not currently running.")\n    return await manager.delete_model(model_name)\n\n\n@app.get("/api/server/code")\nasync def inspect_server_code():\n    return manager.get_runner_code_and_logs()\n\n\n@app.post("/api/server/deploy")\nasync def deploy_server(req: DeployRequest):\n    return StreamingResponse(\n        manager.deploy_and_start(req),\n        media_type="application/x-ndjson",\n        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},\n    )\n\n\n@app.post("/api/server/stop")\nasync def stop_server():\n    return await manager.stop_ollama()\n\n\n@app.post("/api/chat")\nasync def chat_completion(req: ChatRequest):\n    return StreamingResponse(\n        manager.stream_chat(req),\n        media_type="application/x-ndjson",\n        headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"},\n    )\n\n\ndef start_with_ngrok(port: int = BACKEND_PORT, ngrok_token: Optional[str] = None) -> None:\n    global PUBLIC_NGROK_URL\n    token = ngrok_token or os.environ.get("NGROK_AUTHTOKEN")\n    if token:\n        from pyngrok import conf, ngrok\n\n        conf.get_default().auth_token = token\n        ngrok.kill()\n        tunnel = ngrok.connect(addr=port, proto="http")\n        PUBLIC_NGROK_URL = tunnel.public_url\n        os.environ["PUBLIC_NGROK_URL"] = PUBLIC_NGROK_URL\n        print("\\n" + "=" * 72)\n        print("  OLLAMA CLOUD RUNNER - GOOGLE COLAB BACKEND IS LIVE!")\n        print(f"  Ngrok URL (Paste into Web App): {PUBLIC_NGROK_URL}")\n        print(f"  Health Check                  : {PUBLIC_NGROK_URL}/api/status")\n        print("=" * 72 + "\\n")\n    else:\n        print("\\n" + "=" * 72)\n        print(f"  Running locally without ngrok at: http://127.0.0.1:{port}")\n        print("  (Pass --ngrok-token or set NGROK_AUTHTOKEN to expose via ngrok)")\n        print("=" * 72 + "\\n")\n\n    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Start Ollama Cloud Runner Backend")\n    parser.add_argument("--port", type=int, default=BACKEND_PORT)\n    parser.add_argument("--ngrok-token", type=str, default=None)\n    args = parser.parse_args()\n    start_with_ngrok(port=args.port, ngrok_token=args.ngrok_token)\n'}

for filename, content in SERVER_FILES.items():
    (SERVER_PKG / filename).write_text(content, encoding="utf-8")

print("✅ Modular Colab backend package created and verified at /content/ollama_cloud_runner/server")


In [ ]:
# @title Step 3: Start Colab Server & Expose via Ngrok
import asyncio
import getpass
import os
import uvicorn
from pyngrok import conf, ngrok

ngrok_token = ""
try:
    from google.colab import userdata
    ngrok_token = userdata.get("NGROK_AUTHTOKEN") or ""
except Exception:
    pass

if not ngrok_token:
    ngrok_token = getpass.getpass("🔑 Paste your NGROK_AUTHTOKEN (from dashboard.ngrok.com): ").strip()

conf.get_default().auth_token = ngrok_token
ngrok.kill()
tunnel = ngrok.connect(addr=8000, proto="http")
os.environ["PUBLIC_NGROK_URL"] = tunnel.public_url

print("\n" + "=" * 74)
print("🚀 GOOGLE COLAB BACKEND CONNECTED!")
print(f"🔗 Paste this Ngrok URL into your Web App:  {tunnel.public_url}")
print("=" * 74 + "\n")

config = uvicorn.Config("server.main:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()